# Tutorial 2 Exercise Solutions

These solutions correspond to the exercises in `Tutorial_2_MV_Effects_Reference_Voltage.ipynb`.

## 1. Initialisation

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

WORKING_DIRECTORY = Path.cwd()
folder_candidates = [
    WORKING_DIRECTORY,
    WORKING_DIRECTORY / "Tutorial_2_MV_Effects_Reference_Voltage",
    WORKING_DIRECTORY.parent / "Tutorial_2_MV_Effects_Reference_Voltage",
]

for candidate in folder_candidates:
    if (candidate / "data").exists() and (candidate / "results").exists():
        TUTORIAL_DIR = candidate
        break
else:
    raise FileNotFoundError("Could not find the Tutorial 2 folder.")

DATA_DIR = TUTORIAL_DIR / "data"
RESULTS_DIR = TUTORIAL_DIR / "results"
STEPS_PER_DAY = 24 * 60 // 5

def load_matrix(dataset_name, split, file_name):
    path = DATA_DIR / dataset_name / split / file_name
    return pd.read_pickle(path).apply(pd.to_numeric, errors="coerce").to_numpy(np.float32)

V_lv_te = load_matrix("synthetic_data_5_min_LV", "test", "V.pkl")
V_mv_te = load_matrix("synthetic_data_5_min_MV", "test", "V.pkl")
mv_effect_test = V_mv_te - V_lv_te
hours = np.arange(STEPS_PER_DAY) * 5 / 60


## 2. E.1: Inspect the MV effect for one customer

In [ ]:
CUSTOMER_TO_INSPECT = 16
DAY_TO_INSPECT = 1

customer_index = CUSTOMER_TO_INSPECT - 1
start = (DAY_TO_INSPECT - 1) * STEPS_PER_DAY
end = start + STEPS_PER_DAY
customer_effect = mv_effect_test[start:end, customer_index]

print(f"Customer C{CUSTOMER_TO_INSPECT}, test day {DAY_TO_INSPECT}")
print(f"Mean MV effect: {customer_effect.mean():.4f} V")
print(f"Std. MV effect: {customer_effect.std():.4f} V")

plt.figure(figsize=(9, 4))
plt.plot(hours, customer_effect, color="#D55E00")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Hour of day")
plt.ylabel("V_MV - V_LV (V)")
plt.title(f"MV effect for customer C{CUSTOMER_TO_INSPECT}")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 3. E.2: Choose the smallest number of reference meters

In [ ]:
number_of_refs = pd.read_csv(
    RESULTS_DIR / "02_reference_voltage" / "number_of_references.csv"
)

threshold = 0.05
valid_rows = number_of_refs[number_of_refs["test RMSE [V]"] < threshold]
smallest = valid_rows.sort_values("n references").iloc[0]

display(number_of_refs.round(4))
print(f"Smallest number of references below {threshold} V: {int(smallest['n references'])}")
print(f"Reference customers: {smallest['reference customers']}")


## 4. E.3: Compare hosting-capacity error

In [ ]:
hosting = pd.read_csv(
    RESULTS_DIR / "04_mv_aware_hosting_capacity" / "hosting_capacity_four_studies.csv"
)

non_ground_truth = hosting[hosting["error vs ground truth [%]"] != 0].copy()
largest_overestimate = non_ground_truth.sort_values(
    "error vs ground truth [%]", ascending=False
).iloc[0]
closest_to_truth = non_ground_truth.iloc[
    non_ground_truth["error vs ground truth [%]"].abs().argmin()
]

display(hosting.round(3))
print("Largest overestimate:")
print(f"  {largest_overestimate['study']} ({largest_overestimate['error vs ground truth [%]']:.1f}%)")
print("Closest to ground truth, excluding ground truth itself:")
print(f"  {closest_to_truth['study']} ({closest_to_truth['error vs ground truth [%]']:.1f}%)")
